In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
from src.utils.hadoop_config import configure_windows_hadoop
from src.utils.spark_utils import initialize_spark
from src.utils.parquet_io import read_parquet_data
from src.schemas.silver.dim_question_schema import build_dim_question_schema
from src.schemas.silver.dim_exam_schema import build_dim_exam_schema
from pyspark.sql import functions as F

In [3]:
configure_windows_hadoop()

In [4]:
spark = initialize_spark("jup-read-silver")

In [5]:
SILVER_DIR = os.path.join("..", "data", "silver", "topik_documents")

In [6]:
df_question = read_parquet_data(spark, os.path.join(SILVER_DIR, "dim_question"), build_dim_question_schema())

In [7]:
df_exam = read_parquet_data(spark, os.path.join(SILVER_DIR, "dim_exam"), build_dim_exam_schema())

In [8]:
df_qe = df_question.join(F.broadcast(df_exam),on="exam_id",how="inner")

In [20]:
df_question.select("page_number").distinct().show()

+-----------+
|page_number|
+-----------+
|         13|
|         14|
|         18|
|         12|
|         25|
|          6|
|          9|
|         16|
|         20|
|         15|
|         17|
|         10|
|         22|
|          4|
|          7|
|          3|
|          8|
|         24|
|         11|
|         19|
+-----------+
only showing top 20 rows


In [23]:
df_qe.filter(F.col("topik_level")=="TOPIK II").filter(F.col("exam_edition")=="102nd").orderBy(F.col("page_number")).show()

+--------------------+--------------------+-----------+--------------------+-----------------------------------+---------------------------------+--------------------------------+---------------------------+---------+----------------+----------+--------------------+------------+-----------+-----------+------------+
|             exam_id|         question_id|page_number|       question_type|                  raw_question_text|                question_stimulus|       question_vocabulary_items|  question_grammar_patterns|    topic|difficulty_level|confidence|           exam_name|exam_edition|exam_number|topik_level|exam_section|
+--------------------+--------------------+-----------+--------------------+-----------------------------------+---------------------------------+--------------------------------+---------------------------+---------+----------------+----------+--------------------+------------+-----------+-----------+------------+
|db462efc-0f95-415...|f1c7744cb4700e9dc...|      

In [ ]:
df_qe.filter((F.col("exam_edition")=="102nd")&(F.col("page_number")==3) & (F.col("topik_level")=="TOPIK II")).show(10)

+-------+-----------+-----------+-------------+-----------------+-----------------+-------------------------+-------------------------+-----+----------------+----------+---------+------------+-----------+-----------+------------+
|exam_id|question_id|page_number|question_type|raw_question_text|question_stimulus|question_vocabulary_items|question_grammar_patterns|topic|difficulty_level|confidence|exam_name|exam_edition|exam_number|topik_level|exam_section|
+-------+-----------+-----------+-------------+-----------------+-----------------+-------------------------+-------------------------+-----+----------------+----------+---------+------------+-----------+-----------+------------+
+-------+-----------+-----------+-------------+-----------------+-----------------+-------------------------+-------------------------+-----+----------------+----------+---------+------------+-----------+-----------+------------+

